In [0]:
# %pip install -U -qqqq 
# backoff 
# databricks-langchain 
# langgraph==0.5.3 
# uv 
# databricks-agents 
# mlflow-skinny[databricks] 
# chromadb 
# sentence-transformers 
# langchain-huggingface
# langchain-chroma 
# wikipedia 
# faiss-cpu

In [0]:
%pip install -U -q databricks-langchain langchain==0.3.7 faiss-cpu wikipedia langchain-community chromadb langchain-openai tiktoken rank_bm25


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from databricks_langchain import ChatDatabricks, DatabricksEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import RetrievalQA

### Simple RAG

In [0]:
# Retriever Config
MAX_WIKI_DOCS_PER_TOPIC = 10 #TODO: recommend starting with a smaller number for testing purposes
VECTOR_TOP_K = 5 # number of documents to return
EMBEDDING_MODEL = "databricks-bge-large-en" # Embedding model endpoint name

# LLM Config
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-1-8b-instruct"

# Initialize embeddings + LLM
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME, temperature=0.2)


In [0]:
from langchain.document_loaders import WikipediaLoader

loader = WikipediaLoader(query="deep learning", load_max_docs=5)
docs = loader.load()

In [0]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(docs)
print(f"len chunk {len(chunks)} ")

len chunk 68 


In [0]:
from langchain_community.vectorstores import FAISS

embeddings = DatabricksEmbeddings(endpoint=EMBEDDING_MODEL)

# Build FAISS index from your chunks
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

In [0]:
# Initialize embeddings + LLM
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME, temperature=0.2)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})  # 检索Top-3相关片段

In [0]:
prompt = ChatPromptTemplate.from_template("""
Please answer the question based on the following context information.
If the context does not provide relevant information, please directly say:
"Based on the available information, I cannot answer this question."

Context: {context}

Question: {question}

Answer:
""")

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",  # The simplest chain type: stuff all retrieved context into the prompt
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt}
)

# 4. Test query
query = "what is deep learning?"
result = rag_chain.invoke({"query": query})

print(f"Question: {query}")
print(f"Answer: {result['result']}")

Question: what is deep learning?
Answer: Deep learning is a form of machine learning that transforms a set of inputs into a set of outputs via an artificial neural network.


### Conversation RAG

In [0]:
from langchain.memory import ConversationBufferMemory
from langchain.chains import ConversationalRetrievalChain

# 1. Add a memory module on top of standard RAG
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# 2. Build a conversational RAG chain
conversational_rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    combine_docs_chain_kwargs={"prompt": prompt}  # You can use a more complex prompt if needed
)

# 3. Simulate a multi-turn conversation
print("--- First Round of Conversation ---")
result1 = conversational_rag_chain.invoke({"question": "What is deep learning?"})
print(f"User: What is deep learning")
print(f"AI: {result1['answer']}")

print("\n--- Second Round of Conversation (Context-Dependent) ---")
result2 = conversational_rag_chain.invoke({"question": "How to learn deep learning?"})
print(f"User: How to learn deep learning?")
print(f"AI: {result2['answer']}")

--- First Round of Conversation ---


/home/spark-b1756d93-84a5-4d2e-bddb-da/.ipykernel/1388424/command-5881766325413111-95536204:5: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


User: What is deep learning
AI: Deep learning is a form of machine learning that transforms a set of inputs into a set of outputs via an artificial neural network, using a hierarchy of layers to transform input data into a progressively more abstract and composite representation.

--- Second Round of Conversation (Context-Dependent) ---
User: How to learn deep learning?
AI: Based on the available information, I cannot answer this question.

However, I can provide some general information on how to learn deep learning. Deep learning is a complex and rapidly evolving field, and there are many resources available to learn it. Here are some general steps to get started:

1. **Familiarize yourself with machine learning basics**: Understand the fundamentals of machine learning, including supervised and unsupervised learning, regression, classification, and neural networks.
2. **Choose a programming language**: Python is a popular choice for deep learning, and libraries like TensorFlow, Keras

### Corrective RAG, CRAG

工作原理：

检索阶段： 从内部向量存储库获取文档。
评估阶段： 轻量级“评分器”模型为每个文档片段赋予评分（正确/模糊/错误）。
决策门控：
- 正确： 直接进入生成器阶段
- 错误： 丢弃数据并触发外部API（如谷歌搜索或Tavily）
- 合成： 基于验证过的内部数据或新获取的外部数据生成答案

In [0]:
# Conceptual code illustrating the decision logic of CRAG (Corrective RAG)
def corrective_rag_workflow(query, vectorstore, web_search_tool):
    # 1. Initial retrieval
    retrieved_docs = vectorstore.similarity_search(query, k=5)
    
    # 2. Evaluate retrieved results (simulate a lightweight evaluator)
    # In practice, this could be a small trained model or a rule-based scorer
    graded_docs = []
    for doc in retrieved_docs:
        # Simple simulation: score by checking keyword overlap with the query
        relevance_score = naive_relevance_scorer(query, doc.page_content)
        if relevance_score > 0.7:
            graded_docs.append(("correct", doc))
        elif relevance_score > 0.3:
            graded_docs.append(("ambiguous", doc))
        else:
            graded_docs.append(("incorrect", doc))
    
    # 3. Decision gate
    if any(grade == "correct" for grade, _ in graded_docs):
        # If we have qualified documents, use them
        context = "\n".join([d.page_content for g, d in graded_docs if g == "correct"])
        print("[CRAG Decision]: Using internal knowledge base.")
    else:
        # If internal docs are low quality, trigger fallback retrieval
        print("[CRAG Decision]: Insufficient internal knowledge, starting fallback retrieval.")
        context = web_search_tool.search(query)
    
    # 4. Generation
    final_prompt = f"Answer the question based on the following information:\n{context}\n\nQuestion: {query}\nAnswer:"
    return llm.invoke(final_prompt)

# A simplistic relevance scorer
def naive_relevance_scorer(query, doc_content):
    query_words = set(query.lower().split())
    doc_words = set(doc_content.lower().split())
    overlap = len(query_words & doc_words)
    return overlap / max(len(query_words), 1)

# Assume we have a web-connected search tool (e.g., a wrapper around the Tavily Search API)
# web_search_tool = TavilySearch()

In [0]:
class MockWebSearchTool:
    def search(self, query):
        if "deep learning" in query.lower():
            return (
                "Deep learning is a form of machine learning that transforms a set of inputs into a set of outputs via an artificial neural network, using a hierarchy of layers to transform input data into a progressively more abstract and composite representation."
            )
        elif "latest breakthrough" in query.lower():
            return (
                "Recent breakthroughs in deep learning include improvements in "
                "large language models, multimodal systems, and efficient fine-tuning methods."
            )
        else:
            return f"External search result related to: {query}"

In [0]:
class MockDoc:
    def __init__(self, content):
        self.page_content = content

class MockVectorStore:
    def similarity_search(self, query, k=5):
        return [
            MockDoc("Deep learning is a subset of machine learning using neural networks."),
            MockDoc("CNNs and transformers are popular deep learning architectures.")
        ]

class MockWebSearchTool:
    def search(self, query):
        return f"[WEB SEARCH] External info about: {query}"

class MockLLM:
    def invoke(self, prompt):
        return f"[LLM OUTPUT BASED ON CONTEXT]\n{prompt[:200]}..."

# Instantiate mocks
vectorstore = MockVectorStore()
web_search_tool = MockWebSearchTool()
llm = MockLLM()

In [0]:
corrective_rag_workflow(
    "What is deep learning?",
    vectorstore,
    web_search_tool
)

[CRAG Decision]: Insufficient internal knowledge, starting fallback retrieval.


'[LLM OUTPUT BASED ON CONTEXT]\nAnswer the question based on the following information:\n[WEB SEARCH] External info about: What is deep learning?\n\nQuestion: What is deep learning?\nAnswer:...'

In [0]:
print(set("What is deep learning?".lower().split()))
print(set("Deep learning is a subset of machine learning.".lower().split()))
print(naive_relevance_scorer(
    "What is deep learning?",
    "Deep learning is a subset of machine learning."
))

{'what', 'is', 'learning?', 'deep'}
{'machine', 'learning', 'of', 'learning.', 'subset', 'a', 'is', 'deep'}
0.5


### Adaptive RAG

工作原理：

复杂度分析：小型分类器模型对查询进行路由分发。

路径A（无需检索）： 适用于问候语或大型语言模型已掌握的常识性问题。

路径B（标准RAG）： 用于简单的事实查证。

路径C（多步智能体）： 处理需跨多源检索的复杂分析型问题。

适用场景：用户问题复杂度差异大，需要兼顾响应速度与成本。

In [0]:
from langchain_core.runnables import RunnableBranch

# 1. Define the routing function (in practice, this could be a small LLM classifier)
def route_question(query):
    simple_keywords = ["hello", "hi", "who are you"]
    complex_keywords = ["compare", "analyze", "trend", "summarize the past five years"]
    
    if any(kw in query.lower() for kw in simple_keywords):
        return "simple"
    elif any(kw in query.lower() for kw in complex_keywords):
        return "complex"
    else:
        return "standard"

# 2. Define different processing branches
def simple_chain(query):
    """Answer directly without retrieval"""
    return llm.invoke(
        f"Respond in a friendly and concise way to the user's greeting or simple question.\nQuestion: {query}"
    )

def standard_chain(query):
    """Standard RAG pipeline (reuse the existing retriever)"""
    docs = retriever.invoke(query)
    context = "\n".join([d.page_content for d in docs])
    return llm.invoke(
        f"Based on the following context:\n{context}\n\nAnswer the question: {query}"
    )

def complex_chain(query):
    """More complex workflow, e.g., multi-step retrieval or agent usage (simplified here as deeper retrieval)"""
    print("[Adaptive RAG]: Complex question detected. Enabling deep retrieval.")
    docs = retriever.invoke(query, search_kwargs={"k": 10})  # Retrieve more documents
    
    # More advanced logic could be added here (e.g., re-ranking, multi-query expansion, etc.)
    context = "\n---\n".join([d.page_content for d in docs])
    
    return llm.invoke(
        f"Please provide a comprehensive analysis based on the following information:\n{context}\n\nQuestion: {query}"
    )

# 3. Build the adaptive routing chain
branch = RunnableBranch(
    (lambda x: route_question(x["query"]) == "simple",
     lambda x: {"result": simple_chain(x["query"])}),
    
    (lambda x: route_question(x["query"]) == "complex",
     lambda x: {"result": complex_chain(x["query"])}),
    
    lambda x: {"result": standard_chain(x["query"])}  # Default branch
)

# 4. Test (Wikipedia-style)
test_queries = [
    "What is deep learning?",
    "What architectures are commonly used in deep learning?",
    "Summarize the history of deep learning and compare CNNs vs Transformers."
]

def to_text(x):
    return x.content if hasattr(x, "content") else str(x)

for q in test_queries:
    response = branch.invoke({"query": q})
    print(f"Question: {q}")
    print(f"Routing decision: {route_question(q)}")
    print(f"Answer: {to_text(response['result'])[:120]}...\n")

Question: What is deep learning?
Routing decision: standard
Answer: [LLM OUTPUT BASED ON CONTEXT]
Based on the following context:
Deep learning is a form of machine learning that transform...

Question: What architectures are commonly used in deep learning?
Routing decision: simple
Answer: [LLM OUTPUT BASED ON CONTEXT]
Respond in a friendly and concise way to the user's greeting or simple question.
Question:...

Question: Summarize the history of deep learning and compare CNNs vs Transformers.
Routing decision: simple
Answer: [LLM OUTPUT BASED ON CONTEXT]
Respond in a friendly and concise way to the user's greeting or simple question.
Question:...



### Fusion RAG

工作原理：

查询扩展：生成用户问题的3-5种变体。

并行检索：在向量数据库中搜索所有变体。

互补排序融合（RRF）：运用数学公式重新排序结果：

最终排序：在多次检索中排名靠前的文档将被提升至顶部。

适用场景：用户提问模糊、口语化，或问题本身涉及多角度时。

核心：查询扩展 + 倒数排序融合 (RRF)。

In [0]:
"""
Fixed LangChain RAG fusion (Dense FAISS + BM25) + optional contextual compression.

Key fixes:
- Use consistent LangChain imports for v0.2+ (prefer langchain_community for FAISS/BM25).
- BM25Retriever expects Documents; easiest is BM25Retriever.from_documents(chunks).
- RetrievalQA.from_chain_type expects input key "query" by default, but some setups use "question".
  We'll call with {"query": ...} and read output robustly.
- ChatDatabricks must be a Runnable/LC LLM (it is). Ensure you imported it correctly.
- Remove unused numpy import.
"""

from langchain.chains import RetrievalQA
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain.retrievers.ensemble import EnsembleRetriever

from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever

# --- 1) Build retrievers ---
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-1-8b-instruct"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME, temperature=0)

# a) Dense vector retriever (FAISS)
vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# b) Sparse retriever (BM25)
# Best: build BM25 directly from Documents (keeps metadata + avoids text-only pitfalls)
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5

# --- 2) Ensemble / fusion retriever ---
ensemble_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, bm25_retriever],
    weights=[0.5, 0.5],  # adjust as needed
)

# --- 3) Optional: contextual compression (LLM-based extractor) ---
llm_for_compression = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME, temperature=0)
compressor = LLMChainExtractor.from_llm(llm_for_compression)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=ensemble_retriever,
)

# --- 4) RAG chain using fused + compressed retriever ---
rag_chain_fusion = RetrievalQA.from_chain_type(
    llm=llm,  # your main generation LLM (ChatDatabricks/ChatOpenAI/etc.)
    retriever=compression_retriever,
    chain_type="stuff",
    return_source_documents=True,  # helpful for debugging
)

# --- 5) Test ---
query = "what is deep learning"
result = rag_chain_fusion.invoke({"query": query})

# RetrievalQA output key differs across versions: "result" or "answer"
answer = result.get("result") or result.get("answer") or ""
print(f"Query: '{query}'")
print(f"Fusion RAG answer: {str(answer)[:200]}...")

# Optional: inspect which docs were used
src_docs = result.get("source_documents", [])
print(f"\n#Source docs used: {len(src_docs)}")
for i, d in enumerate(src_docs[:3], 1):
    snippet = (d.page_content[:140] + "...") if len(d.page_content) > 140 else d.page_content
    print(f"{i}. {snippet}")

Query: 'what is deep learning'
Fusion RAG answer: Deep learning is a form of machine learning that transforms a set of inputs into a set of outputs via an artificial neural network. It involves using a hierarchy of layers to transform input data into...

#Source docs used: 9
1. Deep reinforcement learning (deep RL) is a subfield of machine learning that combines reinforcement learning (RL) and deep learning. RL cons...
2. Deep learning is a form of machine learning that transforms a set of inputs into a set of outputs via an artificial neural network. Deep lea...
3. >>>
(e.g. every pixel rendered to the screen in a video game) and decide what actions to perform to optimize an objective (e.g. maximizing t...


### HyDE

工作原理：

- 假设生成：大型语言模型为查询编写虚构（假设性）答案。
- 向量化：将虚构答案转化为向量。
- 检索匹配：利用该向量查找与虚构答案风格相似的真实文档。
- 生成最终答案：基于真实文档撰写最终回复。
- 适用场景：问题非常抽象、开放，或与知识库文档表述差异大时。

工作流：问题 → 生成假设答案 → 向量化假设答案 → 用该向量检索 → 生成最终答案。

In [0]:
from langchain_core.messages import AIMessage

def hyde_retrieval_deep_learning(query, vectorstore, llm, embeddings, k=3):
    """
    HyDE for Deep Learning / Wikipedia-style retrieval.

    Args:
        query: user question (e.g., "What is deep learning?")
        vectorstore: your vector DB (FAISS, etc.) built with the SAME embeddings
        llm: ChatDatabricks / ChatOpenAI (must support .invoke and return a message with .content)
        embeddings: the SAME embedding object used to build the vectorstore
        k: number of docs to retrieve
    """

    # 1) Hypothetical Document Generation (HyDE)
    hypothetical_prompt = f"""
You are writing a short Wikipedia-style paragraph that directly answers the question.
Include key definitions, important keywords, and common phrasing used in deep learning literature.
Keep it factual and concise (5–8 sentences).

Question: {query}

Wikipedia-style answer paragraph:
""".strip()

    hypo_resp = llm.invoke(hypothetical_prompt)
    hypothetical_answer = hypo_resp.content if hasattr(hypo_resp, "content") else str(hypo_resp)

    print(f"[HyDE] Hypothetical answer (snippet): {hypothetical_answer[:120]}...")

    # 2) Embed the hypothetical answer using the SAME embedding model as indexing
    hypothetical_embedding = embeddings.embed_query(hypothetical_answer)

    # 3) Retrieve real docs using the hypothetical embedding
    relevant_docs = vectorstore.similarity_search_by_vector(hypothetical_embedding, k=k)

    # 4) Final answer grounded in retrieved docs
    context = "\n\n---\n\n".join([doc.page_content for doc in relevant_docs])

    final_prompt = f"""
Answer the question using ONLY the information in the context below.
If the context does not contain the answer, say: "Based on the provided context, I cannot answer."

Context:
{context}

Question: {query}

Answer:
""".strip()

    final_resp = llm.invoke(final_prompt)
    final_answer = final_resp.content if hasattr(final_resp, "content") else str(final_resp)

    return final_answer, relevant_docs


# ----------------------------
# Example Deep Learning queries
# ----------------------------
test_queries = [
    "What is deep learning?",
    "How does backpropagation work in deep learning?",
    "What are common deep learning architectures (e.g., CNN, RNN, Transformer)?",
]

for q in test_queries:
    answer, docs = hyde_retrieval_deep_learning(
        query=q,
        vectorstore=vectorstore,
        llm=llm,                 # e.g., ChatDatabricks(endpoint=..., temperature=0.2)
        embeddings=embeddings,   # MUST match the embeddings used to build vectorstore
        k=3
    )
    print("\n==============================")
    print("Question:", q)
    print("Answer:", answer[:400], "...")
    print(f"Retrieved docs: {len(docs)}")

[HyDE] Hypothetical answer (snippet): Deep learning is a subset of machine learning that employs artificial neural networks with multiple layers to analyze an...

Question: What is deep learning?
Answer: Deep learning is a form of machine learning that transforms a set of inputs into a set of outputs via an artificial neural network. ...
Retrieved docs: 3
[HyDE] Hypothetical answer (snippet): Backpropagation is a widely used algorithm in deep learning for training artificial neural networks. It is a method for ...

Question: How does backpropagation work in deep learning?
Answer: Based on the provided context, I cannot answer. ...
Retrieved docs: 3
[HyDE] Hypothetical answer (snippet): Deep learning architectures are a set of neural network models that are widely used in various machine learning applicat...

Question: What are common deep learning architectures (e.g., CNN, RNN, Transformer)?
Answer: Fully connected networks, deep belief networks, recurrent neural networks, convolutiona

### Self-RAG

工作原理：

- 检索：由模型自身触发的标准搜索。
- token生成： 模型在生成文本时同步生成特殊标记，如[IsRel]（是否相关？）、[IsSup]（该论点是否得到支持？）和[IsUse]（是否具有实用性？）。
- 自我修正：若模型输出[NoSup]标记，则暂停操作，重新检索信息并重写句子。
- 核心：需要专门微调的模型（如Self-RAG-Llama），在生成流中动态决策是否需要检索以及当前输出的可信度。其流程可以理解为：生成 -> 反思 -> (必要时)检索/重写 -> 继续生成。

In [0]:
# Note: A full Self-RAG system requires fine-tuning the model.
# This example demonstrates the “reflection” logic conceptually.
import re

def extract_claim(text):
    """
    Extract the first claim marked with:
    [Needs support? Yes]
    """
    pattern = r"(.*?)\s*\[Needs support\? Yes\]"
    match = re.search(pattern, text, re.DOTALL)
    
    if match:
        claim = match.group(1).strip()
        return claim
    
    return ""

def self_rag_style_generation(query, retriever, llm):
    max_steps = 3
    context = ""
    
    for step in range(max_steps):
        # Generation phase with built-in "reflection"
        prompt = f"""
        Question: {query}
        Known context: {context}
        
        Please generate the next part of the answer, and after each key claim,
        evaluate whether it requires supporting evidence.
        For example: "The company’s core value is innovation [Needs support? Yes]. ..."
        """
        
        generation_output = llm.invoke(prompt)
        output_text = generation_output.content if hasattr(generation_output, "content") else str(generation_output)
        
        # Simulate parsing reflection markers in the output
        if "[Needs support? Yes]" in output_text:
            print(f"[Self-RAG Step {step+1}]: A claim requiring evidence was detected. Triggering retrieval.")
            
            # Extract the claim that needs verification (placeholder function)
            claim_to_verify = extract_claim(output_text)
            
            # Retrieve supporting documents
            new_docs = retriever.invoke(claim_to_verify)
            context += "\n" + "\n".join([d.page_content for d in new_docs])
            
            # Continue loop: regenerate/refine answer based on updated context
        else:
            print(f"[Self-RAG Step {step+1}]: Generated content appears well-supported. Finishing.")
            return output_text
    
    return "After multiple rounds of retrieval and verification, the final answer is:\n" + output_text


# This is a highly simplified conceptual demonstration.
# In real Self-RAG systems, reflection and retrieval decisions are handled internally by the model.

In [0]:
test_queries = [
    "Who won the 2012 ImageNet competition?",
    "What are the main deep learning architectures?",
    "What is deep learning?",
    "What is the largest neural network ever trained?"
]

for q in test_queries:
    print("\n====================")
    print("Query:", q)
    answer = self_rag_style_generation(q, retriever, llm)
    print("Final Answer:", answer[:400])


Query: Who won the 2012 ImageNet competition?
[Self-RAG Step 1]: Generated content appears well-supported. Finishing.
Final Answer: The 2012 ImageNet Large Scale Visual Recognition Challenge (ILSVRC) was a competition to recognize objects in images. The winner of the competition was AlexNet, a deep neural network developed by a team of researchers led by Alex Krizhevsky, Ilya Sutskever, and Geoffrey Hinton.

Key claim 1: AlexNet was a deep neural network.
[Needs support? No, this is a factual statement about the architecture o

Query: What are the main deep learning architectures?
[Self-RAG Step 1]: Generated content appears well-supported. Finishing.
Final Answer: Here's the next part of the answer:

The main deep learning architectures can be broadly categorized into the following:

1. **Feedforward Neural Networks (FNNs)**: These are the simplest type of deep learning architecture, where the data flows only in one direction, from input layer to output layer, without any feedback lo

### Agentic RAG

### 工作原理：

- 分析阶段：
智能体首先解析用户查询，判定其属于简单查询、多步骤查询、模糊查询或实时数据需求。
- 规划阶段：
将查询分解为子任务并制定策略。
例如：应优先进行向量搜索？网页搜索？调用API？还是提出后续问题？
- 执行：
智能体调用向量数据库、网页搜索、内部API或计算器等工具执行步骤。
- 迭代：
基于中间结果，智能体可优化查询、获取更多数据或验证来源。
- 生成：
收集充分证据后，大型语言模型生成基于事实、感知上下文的最终响应。

In [0]:
from langchain_community.vectorstores import FAISS
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import Tool

# ----------------------------
# 1) Build Wiki vectorstore + retriever
# ----------------------------
wiki_vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)
wiki_retriever = wiki_vectorstore.as_retriever(search_kwargs={"k": 6})

def _run_retriever(query: str) -> str:
    """
    Tool function: returns retrieved passages as a single text blob.
    Compatible with retrievers that support .invoke() or .get_relevant_documents().
    """
    if hasattr(wiki_retriever, "invoke"):
        docs = wiki_retriever.invoke(query)
    else:
        docs = wiki_retriever.get_relevant_documents(query)

    # Keep it simple: return concatenated snippets
    return "\n\n---\n\n".join(d.page_content for d in docs)

wiki_search_tool = Tool(
    name="wiki_search",
    description=(
        "Search the local Wikipedia-derived knowledge base about deep learning. "
        "Use this to look up factual definitions, architectures (CNN/RNN/Transformer), "
        "training (backprop/SGD), history (ImageNet 2012/AlexNet), and limitations."
    ),
    func=_run_retriever,
)

tools = [wiki_search_tool]

# ----------------------------
# 2) Agent prompt (Agentic RAG)
# ----------------------------
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a deep learning assistant. "
     "For factual questions, call the tool `wiki_search` to retrieve evidence first, "
     "then answer grounded in the retrieved text. "
     "If the retrieved text does not contain the answer, say so."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# ----------------------------
# 3) Create tool-calling agent + executor
# ----------------------------
agent = create_tool_calling_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    return_intermediate_steps=True,  # useful for debugging tool calls
)

# ----------------------------
# 4) Test
# ----------------------------
queries = [
    "What is deep learning?",
    "Compare CNNs and Transformers and give typical use cases.",
    "What happened in the 2012 ImageNet competition and why was it important?",
]

for q in queries:
    print("\n" + "=" * 90)
    print("User:", q)
    result = agent_executor.invoke({"input": q})
    print("\nAgent Answer:\n", result["output"])


User: What is deep learning?


> Entering new AgentExecutor chain...

Invoking: `wiki_search` with `deep learning`


Deep learning is a form of machine learning that transforms a set of inputs into a set of outputs via an artificial neural network. Deep learning methods, often using supervised learning with labeled datasets, have been shown to solve tasks that involve handling complex, high-dimensional raw input data (such as images) with less manual feature engineering than prior methods, enabling significant progress in several fields including computer vision and natural language processing. In the past

---

== Overview ==


=== Deep learning ===

---

Fundamentally, deep learning refers to a class of machine learning algorithms in which a hierarchy of layers is used to transform input data into a progressively more abstract and composite representation. For example, in an image recognition model, the raw input may be an image (represented as a tensor of pixels). The first represe

### GraphRAG

工作原理：

- 图构建：
知识被建模为图结构，其中节点代表实体（人物、组织、概念、事件），边代表关系（影响、依赖于、资助于、受监管于）。
- 查询解析：
分析用户查询以识别关键实体和关系类型，而非仅提取关键词。
- 图遍历：
系统遍历图结构，寻找能跨多跳连接实体的有意义路径。
- 可选混合检索：
常结合图结构使用向量搜索，将实体锚定于非结构化文本中。
- 生成：
大型语言模型将发现的关系路径转化为结构化、可解释的答案。

场景：“美联储加息如何通过风险资本影响到我们公司C轮融资的估值？”

工作流：用户查询 → 识别实体和关系 → 在图数据库中遍历路径（如：美联储 → 提高利率 → 影响 → 风险投资意愿 → 影响 → 初创公司估值） → 生成解释性答案。

In [0]:
%pip install neo4j

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from neo4j import GraphDatabase

class Neo4jGraphDB:
    def __init__(self, uri, user, password):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def query_relationship_paths(self, entities):
        if len(entities) < 2:
            return []

        start, end = entities[0], entities[-1]

        query = f"""
        MATCH p=(a:Entity {{name:$start}})-[:AFFECTS*..3]->(b:Entity {{name:$end}})
        RETURN p LIMIT 5
        """

        with self.driver.session() as session:
            result = session.run(query, start=start, end=end)
            return [record["p"] for record in result]

In [0]:
def extract_entities(query, llm):
    prompt = f"""
Extract key entities from the question below.
Return them as a Python list.

Question: {query}
Entities:
"""
    response = llm.invoke(prompt)
    text = response.content if hasattr(response, "content") else str(response)

    try:
        return eval(text)
    except:
        return []

In [0]:
def describe_path(path):
    nodes = [node["name"] for node in path.nodes]
    return " -> ".join(nodes)

In [0]:
def graph_rag_query(query, graph_db, llm):
    # 1. Extract entities
    entities = extract_entities(query, llm)
    print("Extracted entities:", entities)

    # 2. Query graph
    paths = graph_db.query_relationship_paths(entities)

    if not paths:
        return "No relationship paths found."

    # 3. Convert to text
    context = ""
    for path in paths:
        context += "- " + describe_path(path) + "\n"

    # 4. Ask LLM to reason over relationships
    prompt = f"""
Based on the following entity relationship paths:

{context}

Question: {query}

Explain the causal or influence chain clearly.
"""
    response = llm.invoke(prompt)
    return response.content if hasattr(response, "content") else str(response)

In [0]:
graph_db = Neo4jGraphDB(
    uri="bolt://neo4j-host:7687",
    user="neo4j",
    password="your_password"
)

query = "How does Federal Reserve rate hikes affect company valuation?"
answer = graph_rag_query(query, graph_db, llm)

print(answer)